# 🔬 NCT-CRC-HE-100K Dataset Processing Pipeline

**Colorectal Cancer Histopathology Dataset**

## Dataset Info
- **Source**: NCT Heidelberg / Zenodo
- **Images**: 100,000 patches
- **Resolution**: 224×224 (native - no resize needed!)
- **Classes**: 9 tissue types → binary
- **Origin**: 🇩🇪 Germany

## Class Mapping
| Original | Binary | Description |
|----------|--------|-------------|
| TUM | Malignant | Tumor epithelium |
| ADI | Benign | Adipose tissue |
| LYM | Benign | Lymphocytes |
| MUC | Benign | Mucus |
| MUS | Benign | Smooth muscle |
| NORM | Benign | Normal colon mucosa |
| STR | Benign | Cancer-associated stroma |
| BACK | Exclude | Background |
| DEB | Exclude | Debris |

---

## 📦 Cell 1: Install Dependencies

In [ ]:
!apt-get update -qq && apt-get install -qq -y libopenjp2-tools > /dev/null 2>&1
!pip install glymur tqdm --quiet
print("✅ Dependencies installed!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 43.3 MB/s eta 0:00:00
✅ Dependencies installed!


## 📚 Cell 2: Import Libraries

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import shutil
import warnings
warnings.filterwarnings('ignore')

try:
    import glymur
    GLYMUR_AVAILABLE = True
    print("✅ Glymur (JPEG2000) available")
except ImportError:
    GLYMUR_AVAILABLE = False
    print("⚠️ Glymur not available, will use OpenJPEG CLI")

print("✅ Libraries imported!")

✅ Glymur (JPEG2000) available
✅ Libraries imported!


## 🔧 Cell 3: Mount Drive & Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ============================================================================
# 🔧 CONFIGURATION - UPDATE THESE PATHS!
# ============================================================================

class Config:
    """Configuration for NCT-CRC-HE-100K dataset processing"""

    # ⚠️ UPDATE THIS PATH!
    NCT_PATH = '/content/drive/MyDrive/NCT-CRC-HE-100K.zip'  # or folder

    # Extraction directory
    EXTRACTED_DIR = '/content/NCT_extracted'

    # Output directories
    BASE_DIR = '/content/drive/MyDrive/NCT_CRC_PLOS_ONE'
    SPLIT_DIR = f'{BASE_DIR}/NCT_CRC_Split'

    # Processing parameters
    IMG_SIZE = (224, 224)  # Already 224x224!
    RANDOM_SEED = 42
    TEST_SIZE = 0.20
    VAL_SIZE = 0.15

    # JPEG2000 compression ratios
    CR_LEVELS = [1, 15, 25, 50, 100, 500, 1000]

    # Class mapping to binary
    # TUM = Tumor (Malignant)
    # All others (except BACK, DEB) = Benign
    CLASS_MAPPING = {
        'TUM': 'malignant',   # Tumor epithelium
        'ADI': 'benign',      # Adipose tissue
        'LYM': 'benign',      # Lymphocytes
        'MUC': 'benign',      # Mucus
        'MUS': 'benign',      # Smooth muscle
        'NORM': 'benign',     # Normal colon mucosa
        'STR': 'benign',      # Cancer-associated stroma
    }

    # Classes to exclude (not tissue)
    EXCLUDE_CLASSES = ['BACK', 'DEB']

    # All original classes
    ALL_CLASSES = ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']

CONFIG = Config()

# Create directories
os.makedirs(CONFIG.BASE_DIR, exist_ok=True)
os.makedirs(CONFIG.SPLIT_DIR, exist_ok=True)

print("="*60)
print("📁 NCT-CRC-HE-100K Configuration:")
print(f"   Source: {CONFIG.NCT_PATH}")
print(f"   Output: {CONFIG.SPLIT_DIR}")
print(f"   Resolution: {CONFIG.IMG_SIZE} (native!)")
print(f"   Excluded: {CONFIG.EXCLUDE_CLASSES}")
print("="*60)

Mounted at /content/drive
📁 NCT-CRC-HE-100K Configuration:
   Source: /content/drive/MyDrive/NCT-CRC-HE-100K.zip
   Output: /content/drive/MyDrive/NCT_CRC_PLOS_ONE/NCT_CRC_Split
   Resolution: (224, 224) (native!)
   Excluded: ['BACK', 'DEB']


## 📦 Cell 4: Extract Dataset

In [ ]:
print("="*60)
print("📦 LOADING NCT-CRC-HE-100K DATASET")
print("="*60)

# Handle ZIP or folder
if CONFIG.NCT_PATH.endswith('.zip'):
    if os.path.exists(CONFIG.NCT_PATH):
        zip_size = os.path.getsize(CONFIG.NCT_PATH) / (1024**3)
        print(f"\n✅ ZIP found: {zip_size:.2f} GB")

        # Check if already extracted
        if os.path.exists(CONFIG.EXTRACTED_DIR) and len(os.listdir(CONFIG.EXTRACTED_DIR)) > 0:
            print("⚠️ Already extracted. Skipping...")
        else:
            print(f"\n⏳ Extracting (this may take 10-20 minutes)...")
            os.makedirs(CONFIG.EXTRACTED_DIR, exist_ok=True)

            with zipfile.ZipFile(CONFIG.NCT_PATH, 'r') as zip_ref:
                zip_ref.extractall(CONFIG.EXTRACTED_DIR)

            print("✅ Extraction complete!")

        SOURCE_DIR = CONFIG.EXTRACTED_DIR
    else:
        print(f"❌ ZIP not found: {CONFIG.NCT_PATH}")
        SOURCE_DIR = None
else:
    SOURCE_DIR = CONFIG.NCT_PATH

# Explore structure
if SOURCE_DIR and os.path.exists(SOURCE_DIR):
    print(f"\n📂 Contents:")
    for item in os.listdir(SOURCE_DIR):
        item_path = os.path.join(SOURCE_DIR, item)
        if os.path.isdir(item_path):
            n_files = len([f for f in os.listdir(item_path) if f.endswith('.tif')])
            print(f"   📁 {item}/ ({n_files} images)")
        else:
            print(f"   📄 {item}")

📦 LOADING NCT-CRC-HE-100K DATASET

✅ ZIP found: 10.89 GB

⏳ Extracting (this may take 10-20 minutes)...
✅ Extraction complete!

📂 Contents:
   📁 NCT-CRC-HE-100K/ (0 images)


In [ ]:
# ============================================================================
# 🔧 FIX: Find the correct source directory with class folders
# ============================================================================

print("🔍 Exploring NCT-CRC-HE-100K structure...")

# Find the folder containing the class subfolders
SOURCE_DIR = CONFIG.EXTRACTED_DIR

for root, dirs, files in os.walk(CONFIG.EXTRACTED_DIR):
    # Check if this folder contains the expected class folders
    expected_classes = ['ADI', 'TUM', 'NORM', 'STR', 'MUS']
    matches = [c for c in expected_classes if c in dirs]

    if len(matches) >= 3:  # Found at least 3 class folders
        SOURCE_DIR = root
        print(f"✅ Found class folders at: {SOURCE_DIR}")
        break

# List the contents
print(f"\n📂 Contents of {SOURCE_DIR}:")
for item in sorted(os.listdir(SOURCE_DIR)):
    item_path = os.path.join(SOURCE_DIR, item)
    if os.path.isdir(item_path):
        # Count images (TIF or PNG)
        n_images = len([f for f in os.listdir(item_path) if f.endswith(('.tif', '.png', '.jpg'))])
        print(f"   📁 {item}/ ({n_images:,} images)")
    else:
        print(f"   📄 {item}")

# Update the class_dirs dictionary
class_dirs = {}
for class_name in CONFIG.ALL_CLASSES:
    class_path = os.path.join(SOURCE_DIR, class_name)
    if os.path.exists(class_path):
        n_images = len([f for f in os.listdir(class_path) if f.endswith(('.tif', '.png', '.jpg'))])
        if n_images > 0:
            class_dirs[class_name] = class_path
            print(f"   ✅ {class_name}: {n_images:,} images")

print(f"\n✅ Found {len(class_dirs)} class folders with images")

## 🔍 Cell 5: Find Image Directories

In [ ]:
print("="*60)
print("🔍 FINDING IMAGE DIRECTORIES")
print("="*60)

# NCT structure: NCT-CRC-HE-100K/ADI/, NCT-CRC-HE-100K/TUM/, etc.
class_dirs = {}

for class_name in CONFIG.ALL_CLASSES:
    # Search for class folder
    for root, dirs, files in os.walk(SOURCE_DIR):
        if class_name in dirs:
            class_path = os.path.join(root, class_name)
            n_images = len([f for f in os.listdir(class_path) if f.endswith(('.tif', '.png', '.jpg'))])
            if n_images > 0:
                class_dirs[class_name] = class_path
                break
        # Check if current folder is the class
        if os.path.basename(root) == class_name:
            n_images = len([f for f in files if f.endswith(('.tif', '.png', '.jpg'))])
            if n_images > 0:
                class_dirs[class_name] = root
                break

print("\n📊 Found class directories:")
total_images = 0
total_included = 0

for class_name in CONFIG.ALL_CLASSES:
    if class_name in class_dirs:
        class_path = class_dirs[class_name]
        n_images = len([f for f in os.listdir(class_path) if f.endswith(('.tif', '.png', '.jpg'))])

        if class_name in CONFIG.EXCLUDE_CLASSES:
            status = "❌ EXCLUDED"
        else:
            binary_class = CONFIG.CLASS_MAPPING[class_name]
            status = f"→ {binary_class}"
            total_included += n_images

        print(f"   {class_name}: {n_images:,} images {status}")
        total_images += n_images
    else:
        print(f"   {class_name}: NOT FOUND")

print(f"\n   Total images: {total_images:,}")
print(f"   After excluding BACK/DEB: {total_included:,}")

# Sample image
sample_class = 'TUM' if 'TUM' in class_dirs else list(class_dirs.keys())[0]
sample_dir = class_dirs[sample_class]
sample_files = [f for f in os.listdir(sample_dir) if f.endswith(('.tif', '.png', '.jpg'))]
if sample_files:
    sample_img = Image.open(os.path.join(sample_dir, sample_files[0]))
    print(f"\n🖼️ Sample image ({sample_class}):")
    print(f"   Size: {sample_img.size}")
    print(f"   Mode: {sample_img.mode}")

🔍 FINDING IMAGE DIRECTORIES

📊 Found class directories:
   ADI: 10,407 images → benign
   BACK: 10,566 images ❌ EXCLUDED
   DEB: 11,512 images ❌ EXCLUDED
   LYM: 11,557 images → benign
   MUC: 8,896 images → benign
   MUS: 13,536 images → benign
   NORM: 8,763 images → benign
   STR: 10,446 images → benign
   TUM: 14,317 images → malignant

   Total images: 100,000
   After excluding BACK/DEB: 77,922

🖼️ Sample image (TUM):
   Size: (224, 224)
   Mode: RGB


## 📂 Cell 6: Create Dataset with Binary Labels

In [ ]:
print("="*60)
print("📂 CREATING DATASET WITH BINARY LABELS")
print("="*60)

all_images = []

for class_name, class_path in class_dirs.items():
    # Skip excluded classes
    if class_name in CONFIG.EXCLUDE_CLASSES:
        print(f"   Skipping {class_name} (excluded)")
        continue

    binary_class = CONFIG.CLASS_MAPPING[class_name]

    image_files = [f for f in os.listdir(class_path) if f.endswith(('.tif', '.png', '.jpg'))]
    print(f"   Processing {class_name}: {len(image_files):,} images → {binary_class}")

    for filename in image_files:
        all_images.append({
            'filepath': os.path.join(class_path, filename),
            'filename': filename,
            'original_class': class_name,
            'binary_class': binary_class,
            'class_idx': 1 if binary_class == 'malignant' else 0,
            'patient_id': filename.replace('.tif', '').replace('.png', '').replace('.jpg', '')
        })

df = pd.DataFrame(all_images)

print(f"\n📊 Dataset Summary:")
print(f"   Total images: {len(df):,}")
print(f"\n   Original class distribution:")
print(df['original_class'].value_counts())
print(f"\n   Binary class distribution:")
print(df['binary_class'].value_counts())

📂 CREATING DATASET WITH BINARY LABELS
   Processing ADI: 10,407 images → benign
   Skipping BACK (excluded)
   Skipping DEB (excluded)
   Processing LYM: 11,557 images → benign
   Processing MUC: 8,896 images → benign
   Processing MUS: 13,536 images → benign
   Processing NORM: 8,763 images → benign
   Processing STR: 10,446 images → benign
   Processing TUM: 14,317 images → malignant

📊 Dataset Summary:
   Total images: 77,922

   Original class distribution:
original_class
TUM     14317
MUS     13536
LYM     11557
STR     10446
ADI     10407
MUC      8896
NORM     8763
Name: count, dtype: int64

   Binary class distribution:
binary_class
benign       63605
malignant    14317
Name: count, dtype: int64


## 📂 Cell 7: Create Train/Val/Test Split

In [ ]:
print("="*60)
print("📂 CREATING TRAIN/VAL/TEST SPLIT")
print("="*60)

# Stratified split by binary class
train_val_df, test_df = train_test_split(
    df,
    test_size=CONFIG.TEST_SIZE,
    stratify=df['binary_class'],
    random_state=CONFIG.RANDOM_SEED
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=CONFIG.VAL_SIZE / (1 - CONFIG.TEST_SIZE),
    stratify=train_val_df['binary_class'],
    random_state=CONFIG.RANDOM_SEED
)

# Add split labels
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

all_splits_df = pd.concat([train_df, val_df, test_df])

print(f"\n📊 Split Results:")
print("-" * 60)
print(f"{'Split':<10} {'Total':<12} {'Benign':<15} {'Malignant':<15}")
print("-" * 60)

for split in ['train', 'val', 'test']:
    split_data = all_splits_df[all_splits_df['split'] == split]
    n_benign = len(split_data[split_data['binary_class'] == 'benign'])
    n_malignant = len(split_data[split_data['binary_class'] == 'malignant'])
    print(f"{split:<10} {len(split_data):<12,} {n_benign:<15,} {n_malignant:<15,}")

print("-" * 60)

# Save metadata
metadata_path = f"{CONFIG.BASE_DIR}/nct_crc_metadata.csv"
all_splits_df.to_csv(metadata_path, index=False)
print(f"\n💾 Metadata saved: {metadata_path}")

📂 CREATING TRAIN/VAL/TEST SPLIT

📊 Split Results:
------------------------------------------------------------
Split      Total        Benign          Malignant      
------------------------------------------------------------
train      50,648       41,343          9,305          
val        11,689       9,541           2,148          
test       15,585       12,721          2,864          
------------------------------------------------------------

💾 Metadata saved: /content/drive/MyDrive/NCT_CRC_PLOS_ONE/nct_crc_metadata.csv


## 🖼️ Cell 8: Copy Original Images (Already 224×224)

In [ ]:
print("="*60)
print("🖼️ COPYING ORIGINAL IMAGES")
print("   (Converting TIF to PNG, already 224×224)")
print("="*60)

# Create directory structure
original_dir = f"{CONFIG.SPLIT_DIR}/CR1_original"
for split in ['train', 'val', 'test']:
    for binary_class in ['benign', 'malignant']:
        os.makedirs(f"{original_dir}/{split}/{binary_class}", exist_ok=True)

# Process images (convert TIF to PNG)
print(f"\n⏳ Processing {len(all_splits_df):,} images...")
print("   (This may take 20-30 minutes for 100k images)")

success_count = 0
for _, row in tqdm(all_splits_df.iterrows(), total=len(all_splits_df)):
    src_path = row['filepath']
    split = row['split']
    binary_class = row['binary_class']

    # New filename: class_originalname.png
    base_name = os.path.splitext(row['filename'])[0]
    new_filename = f"{row['original_class']}_{base_name}.png"
    dst_path = f"{original_dir}/{split}/{binary_class}/{new_filename}"

    try:
        img = Image.open(src_path).convert('RGB')
        img.save(dst_path, 'PNG')
        success_count += 1
    except Exception as e:
        pass

print(f"\n✅ Processed {success_count:,}/{len(all_splits_df):,} images")

# Verify
print(f"\n📊 Verification:")
for split in ['train', 'val', 'test']:
    for cls in ['benign', 'malignant']:
        path = f"{original_dir}/{split}/{cls}"
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"   {split}/{cls}: {count:,} images")

🖼️ COPYING ORIGINAL IMAGES
   (Converting TIF to PNG, already 224×224)

⏳ Processing 77,922 images...
   (This may take 20-30 minutes for 100k images)


100%|██████████| 77922/77922 [50:59<00:00, 25.47it/s]



✅ Processed 77,922/77,922 images

📊 Verification:
   train/benign: 41,343 images
   train/malignant: 9,305 images
   val/benign: 9,541 images
   val/malignant: 2,148 images
   test/benign: 12,721 images
   test/malignant: 2,864 images


## 🗜️ Cell 9: Apply JPEG2000 Compression

In [ ]:
print("="*60)
print("🗜️ APPLYING JPEG2000 COMPRESSION")
print("="*60)
print("\n⚠️ This will take 2-4 hours for 100k images × 6 CR levels")

def compress_image_jpeg2000(input_path, output_path, compression_ratio):
    """Compress image using JPEG2000"""
    try:
        if GLYMUR_AVAILABLE:
            img = np.array(Image.open(input_path).convert('RGB'))
            temp_jp2 = output_path.replace('.png', '.jp2')

            glymur.Jp2k(temp_jp2, img, cratios=[compression_ratio])

            jp2 = glymur.Jp2k(temp_jp2)
            img_decoded = Image.fromarray(jp2[:])
            img_decoded.save(output_path, 'PNG')

            if os.path.exists(temp_jp2):
                os.remove(temp_jp2)

            return True
        else:
            import subprocess
            temp_j2k = output_path.replace('.png', '.j2k')

            subprocess.run([
                'opj_compress', '-i', input_path, '-o', temp_j2k,
                '-r', str(compression_ratio), '-p', 'RPCL', '-SOP', '-EPH'
            ], capture_output=True, check=True)

            subprocess.run([
                'opj_decompress', '-i', temp_j2k, '-o', output_path
            ], capture_output=True, check=True)

            if os.path.exists(temp_j2k):
                os.remove(temp_j2k)

            return os.path.exists(output_path)

    except Exception as e:
        return False

# Process each compression level
for cr in CONFIG.CR_LEVELS[1:]:  # Skip CR1
    cr_dir = f"{CONFIG.SPLIT_DIR}/CR{cr}"
    print(f"\n{'='*60}")
    print(f"📊 Processing CR{cr}...")
    print(f"{'='*60}")

    # Create directories
    for split in ['train', 'val', 'test']:
        for binary_class in ['benign', 'malignant']:
            os.makedirs(f"{cr_dir}/{split}/{binary_class}", exist_ok=True)

    # Compress all images
    original_base = f"{CONFIG.SPLIT_DIR}/CR1_original"
    success_count = 0
    total_count = 0

    for split in ['train', 'val', 'test']:
        for binary_class in ['benign', 'malignant']:
            src_dir = f"{original_base}/{split}/{binary_class}"
            dst_dir = f"{cr_dir}/{split}/{binary_class}"

            if not os.path.exists(src_dir):
                continue

            files = [f for f in os.listdir(src_dir) if f.endswith('.png')]

            for img_file in tqdm(files, desc=f"{split}/{binary_class}", leave=False):
                src_path = f"{src_dir}/{img_file}"
                dst_path = f"{dst_dir}/{img_file}"
                total_count += 1

                if compress_image_jpeg2000(src_path, dst_path, cr):
                    success_count += 1

    print(f"   ✅ Compressed {success_count:,}/{total_count:,} images at CR{cr}")

print("\n" + "="*60)
print("✅ ALL COMPRESSION COMPLETE!")
print("="*60)

🗜️ APPLYING JPEG2000 COMPRESSION

⚠️ This will take 2-4 hours for 100k images × 6 CR levels

📊 Processing CR15...


   ✅ Compressed 77,922/77,922 images at CR15

📊 Processing CR25...


   ✅ Compressed 77,922/77,922 images at CR25

📊 Processing CR50...


   ✅ Compressed 77,922/77,922 images at CR50

📊 Processing CR100...


   ✅ Compressed 77,922/77,922 images at CR100

📊 Processing CR500...


   ✅ Compressed 77,922/77,922 images at CR500

📊 Processing CR1000...


   ✅ Compressed 77,922/77,922 images at CR1000

✅ ALL COMPRESSION COMPLETE!


## 🔍 Cell 10: Verify Final Dataset

In [ ]:
print("="*60)
print("🔍 FINAL DATASET VERIFICATION")
print("="*60)

print("\n📊 Image counts per compression level:")
print("-" * 80)
print(f"{'CR Level':<15} {'Train':<15} {'Val':<15} {'Test':<15} {'Total':<15}")
print("-" * 80)

for cr in ['CR1_original'] + [f'CR{c}' for c in CONFIG.CR_LEVELS[1:]]:
    cr_dir = f"{CONFIG.SPLIT_DIR}/{cr}"

    counts = {'train': 0, 'val': 0, 'test': 0}
    for split in counts.keys():
        for binary_class in ['benign', 'malignant']:
            class_dir = f"{cr_dir}/{split}/{binary_class}"
            if os.path.exists(class_dir):
                counts[split] += len([f for f in os.listdir(class_dir) if f.endswith('.png')])

    total = sum(counts.values())
    status = "✅" if total > 0 else "❌"
    print(f"{status} {cr:<13} {counts['train']:<15,} {counts['val']:<15,} {counts['test']:<15,} {total:<15,}")

print("-" * 80)

# Show sample comparison
print("\n🖼️ Sample Image Comparison:")
sample_file = None
test_dir = f"{CONFIG.SPLIT_DIR}/CR1_original/test/malignant"
if os.path.exists(test_dir) and os.listdir(test_dir):
    sample_file = os.listdir(test_dir)[0]

if sample_file:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    cr_show = ['CR1_original', 'CR50', 'CR100', 'CR500']

    for i, cr in enumerate(cr_show):
        img_path = f"{CONFIG.SPLIT_DIR}/{cr}/test/malignant/{sample_file}"
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[i].imshow(img)
            axes[i].set_title(cr.replace('CR1_original', 'Original'))
            axes[i].axis('off')

    plt.suptitle('NCT-CRC Sample: Compression Comparison (Native 224×224)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{CONFIG.BASE_DIR}/nct_crc_compression_sample.png", dpi=150)
    plt.show()

print("\n" + "="*60)
print("🎉 NCT-CRC-HE-100K DATASET READY!")
print("="*60)
print(f"\n📁 Location: {CONFIG.SPLIT_DIR}")
print(f"\n📊 Tissue types included:")
print(f"   Malignant: TUM (Tumor epithelium)")
print(f"   Benign: ADI, LYM, MUC, MUS, NORM, STR")